# Parlay Auto Devig

This notebook demonstrates an image-to-odds workflow. A locally hosted Gemma 4 12B model receives a parlay screenshot through Ollama over Tailscale and extracts structured bet details. Those details drive an odds database search. Laya, a non-autoregressive system 1 decision model, quickly narrows the results by distinguishing full-game from partial-game markets, choosing the matching market, and resolving ambiguous events. The selected event and market are then used to retrieve both sides of the odds needed to devig a leg.

In [28]:
import base64
import httpx
import json
import os
import pandas as pd

from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from laya import Router
from pandas import DataFrame
from pathlib import Path

load_dotenv()

import warnings
from tqdm.std import TqdmWarning

warnings.filterwarnings(
    "ignore",
    message=r"IProgress not found.*",
    category=TqdmWarning,
)

warnings.filterwarnings(
    "ignore",
    message=r"laya: this checkpoint ships invalid temperatures.*",
    category=RuntimeWarning,
)

In [14]:
EXTRACTION_PROMPT = """You are a precise data-extraction tool for sports betting slips.

Read the bet slip in the image and return ONLY a JSON object matching the schema below. No prose, no markdown fences.

Top level:
- "sportsbook": the book's name if visible (e.g. "DraftKings", "FanDuel"), else null.
- "bet_type": "single", "parlay", or "sgp". Use "sgp" when every leg comes from one game.
- "stake": amount wagered, as a number with no currency symbol. null if not shown.
- "payout": total potential return including stake, as a number. null if not shown.
- "final_odds": the slip's overall American odds, signed (e.g. "+850"). null if not shown.
- "boost_from": on a boosted slip, the original pre-boost price. null otherwise.
- "legs": one object per selection, in the order printed.

There is no slip-wide league: a parlay can span games, sports and leagues at
once, so only ask for league on each leg below.

Each leg:
- "raw_text": the selection text exactly as printed, verbatim.
- "selection": only the player or team, with no rank, matchup, line, or side (e.g. "Jayson Tatum", "Boston Celtics"). null for a game total with no selected team or player.
- "market_raw": the market label as printed (e.g. "Points", "Pts+Reb+Ast", "Spread", "Moneyline").
- "sport": that leg's sport: "Baseball", "Basketball", "Football", "Hockey", "Soccer", "Tennis", or "MMA". null only if it cannot be determined.
- "league": that leg's own league (NBA, NFL, MLB, NHL, NCAAB, NCAAF, NCAAW, WNBA), else null.
- "side": "Over", "Under", "Yes", "No", or null.
- "line": the numeric line as a number, else null.
- "odds": that leg's American odds, signed (e.g. "-110"). null if not shown.
- "event": the matchup if printed (e.g. "Boston Celtics @ Golden State Warriors"), else null.

Rules:
- "25+ Points" means Over 24.5, so side "Over" and line 24.5. Any "N+" is Over (N minus 0.5).
- "Over 220.5" means side "Over", line 220.5.
- "Celtics -5.5" means market_raw "Spread", selection "Boston Celtics", line -5.5, side null.
- Spreads keep their sign in "line" (-5.5, +3.5). Totals are positive.
- Odds always carry an explicit sign. "EVEN" or "EV" means "+100".
- Write player and team names exactly as printed. Do not abbreviate and do not guess at expansions.
- A parenthesized matchup code after a selection, such as "Florida to Win (MISS @ FLA)",
  belongs in "event" as "MISS @ FLA". Keep "selection" as the printed team name.
- A ranking such as "#8" before a team is not part of "selection":
  "#8 Alabama to Win (SCAR @ ALA)" has selection "Alabama" and event "SCAR @ ALA".
- If a value is unreadable or absent, use null. Never invent one.
- Return every leg you can see, even when some of its fields are null.

Where the prices live:
- The slip's overall price is the large one shown once, away from the legs —
  usually in the header or in a coloured bar at the bottom. Read it carefully;
  it is the single most important number on the slip.
- A price with a line struck through it is the OLD price. Two prices joined by
  an arrow or chevron, like "+535 > +661", mean boosted: the struck-out left
  value is "boost_from" and the right value is "final_odds".
- "$10 pays $76.14" means stake 10 and payout 76.14. Fill in both.
- Leg odds are only the small prices printed beside an individual leg. Many
  slips, boosts especially, print no per-leg prices at all — then every leg's
  "odds" is null. Never copy the slip's overall price, or another leg's price,
  into a leg.

Reading the market:
- Use the market label printed under or beside the selection, verbatim, however
  long: "GAME RESULT (90 MINUTES + STOPPAGE TIME)", "HITS+RUNS+RBI".
- When no separate label is printed, take the market from the wording:
  "Yankees to Win" is market_raw "Moneyline", "Ben Rice 1+ Hits" is "Hits".
- Promo names and badges are not markets and not sportsbooks. "ODDS BOOST",
  "Sunshine State", "Play of the Day" and similar headline text name the
  promotion; unless the book's own name is visible, "sportsbook" is null.
- "sport" describes the game even when "league" is null. For example, use
  sport "Soccer" and league null for MLS, or sport "Tennis" and league null.
- "league" must be one of the leagues listed above or null. Soccer, tennis,
  MMA and anything else outside that list is null — do not substitute a listed league.
- Use "sgp" only when every leg is from the same one game. Legs from different
  games are "parlay", even under a single promo heading.
"""

In [15]:
LEG_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "raw_text", "selection", "market_raw", "sport", "league",
        "side", "line", "odds", "event",
    ],
    "properties": {
        "raw_text": {"type": "string"},
        "selection": {"type": ["string", "null"]},
        "market_raw": {"type": "string"},
        "sport": {
            "type": ["string", "null"],
            "enum": [
                "Baseball", "Basketball", "Football", "Hockey",
                "Soccer", "Tennis", "MMA", None,
            ],
        },
        "league": {
            "type": ["string", "null"],
            "enum": [
                "NBA", "NFL", "MLB", "NHL", "NCAAB", "NCAAF",
                "NCAAW", "WNBA", None,
            ],
        },
        "side": {"type": ["string", "null"]},
        "line": {"type": ["number", "null"]},
        "odds": {"type": ["string", "null"]},
        "event": {"type": ["string", "null"]},
    },
}

In [16]:
EXTRACTION_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "required": [
        "sportsbook", "bet_type",
        "stake", "payout", "final_odds", "boost_from", "legs",
    ],
    "properties": {
        "sportsbook": {"type": ["string", "null"]},
        "bet_type": {"type": ["string", "null"]},
        "stake": {"type": ["number", "null"]},
        "payout": {"type": ["number", "null"]},
        "final_odds": {"type": ["string", "null"]},
        "boost_from": {"type": ["string", "null"]},
        "legs": {"type": "array", "items": LEG_SCHEMA},
    },
}

In [ ]:
def ollama_process_image(ollama: str, model: str, image):
    payload = {
        "model": model,
        "messages": [{
            "role": "user",
            "content": EXTRACTION_PROMPT,
            "images": [base64.b64encode(image.read_bytes()).decode("ascii")],
        }],
        "format": EXTRACTION_SCHEMA,
        "stream": False,
        "think": False,
        "keep_alive": "10m",
        "options": {"temperature": 0, "seed": 0, "num_predict": 2048},
    }

    headers = {"Host": "localhost:11434"}

    response = httpx.post(
        f"{ollama}/api/chat", json=payload, headers=headers, timeout=180
    )
    response.raise_for_status()
    body = response.json()


    result = json.loads(body["message"]["content"])

    for i, leg in enumerate(result["legs"], 1):
        print(f"Leg {i}: {leg['raw_text']}")
        print(f"   selection={leg['selection']} market={leg['market_raw']!r} "
            f"league={leg['league']!r} side={leg['side']!r} line={leg['line']!r} "
            f"odds={leg['odds']!r} event={leg['event']!r}")

    return result


def search_for_event(api_token: str, books: list[str], search_text: str, league_id: str = None) -> DataFrame:
    now = datetime.now(timezone.utc)
    URL = f"{os.getenv('ODDS_SERVICE_URL')}/v1/markets/discover"

    query = {
        "selection_search": search_text,
        "sportsbook_ids": books,
        "starts_from": (now).isoformat(),   # required with selection_search
        "starts_to": (now + timedelta(days=7)).isoformat(),      # max 14-day window
    }
    if league_id:
        query["league_id"] = league_id

    response = httpx.post(
        url=URL,
        headers={"Authorization": f"Bearer {api_token}"},
        json={
            "queries": [query],
            "freshness": {
                "max_acquisition_age_seconds": 3600,   # 30 hides most data; loosen to explore
                "max_upstream_age_seconds": None,
                "require_upstream_evidence": False,
                "include_retained": False,
            },
        },
        timeout=30,
    )
    response.raise_for_status()
    results = response.json()["results"]

    rows = []
    for r in results:
        e = r["event"]
        for m in r["markets"]:
            rows.append({
                "event": f"{" @ ".join(e['participants'])}",
                "starts_at": e["starts_at"],
                "league": e["league_id"],
                "event_id": e["provider_event_id"],
                "state": e["state"],
                "book": r["source"]["sportsbook_id"],
                "book_status": r["availability"],
                "book_health": r["source_health"],
                "market": m["market"],
                "period": m["period"],
                "selection": m["selection"],
                "side": m["side"],
                "line": m["line"],
                "is_main": m["is_main"],
                "availability": m["availability"],
                "provider": m["provider"],
                "odds_id": m["provider_odds_id"],
            })

    df = pd.DataFrame(rows)
    df["starts_at"] = pd.to_datetime(df["starts_at"])
    df["line"] = pd.to_numeric(df["line"])
    pd.set_option("display.max_rows", 200, "display.max_colwidth", 40)

    return df


def laya_noul_partial_game_bet(router: Router,  market_raw: str) -> dict[str,any]:
    router = Router()
    state = f"{market_raw}"
    questions = {
        "full_game": {
            "type": "noul",
            "instructions": "Is the bet market for only an inning, quarter, or half of the game?",
        },
    }
    laya = router.predict(state, questions)
    return laya


def laya_process_search_results(router: Router, market_raw: str, markets: dict[str,any]) -> dict[str,any]:
    state = f"{market_raw}"
    questions = {
        "market": {
            "type": "choice",
            "instructions": f"Which market is the best match?",
             "criteria": markets
        },
    }
    laya = router.predict(state, questions)  # model="english", max_len=1024, head_max_len=512
    return laya


def laya_process_search_event(router: Router, game: str, events: list[str]) -> dict[str,any]:
    state = f"We are attending a sports event with the following abbreviated name: '{game}'"
    questions = {
        "game": {
            "type": "choice",
            "instructions": f"Which event best matches?",
             "criteria": events
        }
    }
    laya_result = router.predict(state, questions)
    return laya_result 


def fetch_leg_complete_odds(api_token: str, books: list[str], event_id: str, market: str) -> DataFrame:
    response = httpx.post(
        f"{os.getenv('ODDS_SERVICE_URL')}/v1/odds/query",
        headers={"Authorization": f"Bearer {api_token}"},
        json={
            "structured": [{
                "event_id": event_id,
                "sportsbook_ids": books,
                "market": market,
                # no "selection" -> both teams come back, which devig needs
                # no "line" for moneyline; see the note below for spreads
            }],
            "freshness": {
                "max_acquisition_age_seconds": 3600,
                "max_upstream_age_seconds": None,
                "require_upstream_evidence": False,
                "include_retained": False,
            },
            "include_diagnostics": True,
        },
        timeout=30,
    )
    response.raise_for_status()

    price_rows = []
    for r in response.json()["results"]:
        q = r.get("quote")
        if not q:
            continue
        price_rows.append({
            "book": q["sportsbook_id"],
            "selection": q["selection"],
            "side": q["side"],
            "line": q["line"],
            "is_main": q["is_main"],
            "price": q["price"],
            "availability": r["availability"],
            "acquired_at": (r.get("publication") or {}).get("acquired_at"),
            "source_health": r.get("source_health"),
            "odds_id": q["provider_odds_id"],
        })

    prices = pd.DataFrame(price_rows)
    prices["price"] = pd.to_numeric(prices["price"])
    prices.sort_values(["book", "selection"])

    return prices


def fetch_markets_glossary(api_token: str, sport: str = None, league_id: str = None) -> dict[str, str]:
    url = f"{os.getenv('ODDS_SERVICE_URL')}/v1/markets/glossary"
    params = {}
    if sport:
        params["sport"] = sport
    if league_id:

        params["league_id"] = league_id
    response = httpx.get(
        url,
        params=params,
        headers={"Authorization": f"Bearer {api_token}"},
        timeout=10,
    )
    response.raise_for_status()
    markets = response.json()["markets"]

    return markets


def fetch_markets_and_descriptions(markets: list[str], markets_glossary: dict[str, str]) -> dict[str, str]:
    matched_markets = {}
    for market in markets:
        glossary = markets_glossary.get(market)
        if glossary:
            # print(f"{market}: {markets_glossary[market]}")
            matched_markets[market] = glossary

    return matched_markets


## Step 0: Benchmark Models on Example Images

I ran two small benchmarks to compare local and API-based image models on extraction accuracy, speed, and cost.

### Local Image Extraction LLM Benchmark
Accuracy is the reported `correct / total` score. Average time is measured per completed image and depends on the hardware used.

| Rank | Model | Images | Accuracy | Correct / Total | Avg. Time / Image |
|---:|---|---:|---:|---:|---:|
| 1 | gemma4:31b | 18/18 | 92.7% | 601/648 | 143.2s |
| 2 | qwen3.8:27b | 18/18 | 91.0% | 590/648 | 75.4s |
| 3 | gemma4:26b | 18/18 | 90.6% | 587/648 | 12.6s |
| 4 | muse-glimmer:30b | 18/18 | 88.3% | 572/648 | 83.1s |
| **5** | **gemma4:12b** | **18/18** | **87.5%** | **567/648** | **28.0s** |
| 6 | qwen2.5vl:7b | 18/18 | 85.0% | 551/648 | 11.0s |
| 7 | gemma3:27b | 18/18 | 82.7% | 536/648 | 124.0s |
| 8 | ministral-3:14b | 18/18 | 81.5% | 528/648 | 45.8s |
| 8 | qwen3.5:9b | 18/18 | 80.6% | 522/648 | 13.3s |
| 9 | gemma3:12b | 18/18 | 79.9% | 518/648 | 36.8s |
| 10 | ministral-3:8b | 18/18 | 79.3% | 521/657 | 20.2s |
| 11 | minicpm-v4.5:8b | 18/18 | 76.7% | 497/648 | 18.1s |
| 12 | ornith-1.5:9b | 18/18 | 74.7% | 484/648 | 8.2s |
| 13 | gemma3:4b | 18/18 | 71.6% | 464/648 | 6.3s |
| 14 | minicpm-v:8b | 18/18 | 62.7% | 406/648 | 6.7s |
| 15 | minicpm-v4.6:1b | 18/18 | 50.3% | 326/648 | 2.5s |
| 16 | gemma4:e4b | 18/18 | 19.0% | 132/693 | 3.9s |


### Cloud-based LLM Image Extraction Model Benchmark

Accuracy is the reported `correct / total` score. Costs are estimated from the listed token prices in the benchmark results.

| Rank | Model | Images | Accuracy | Correct / Total | Avg. Time / Image | Total Test Time | Listed Cost | Listed Cost / Image |
|---:|---|---:|---:|---:|---:|---:|---:|---:|
| 1 | deepseek-v4-1-flash | 18/18 | 93.4% | 605/648 | 3.3s | 59.2s | $0.0123 | $0.00068 |
| 2 | grok-4-7 | 18/18 | 93.1% | 603/648 | 9.7s | 174.5s | $0.1973 | $0.01096 |
| 3 | kimi-k2-6 | 18/18 | 92.6% | 600/648 | 8.1s | 146.1s | $0.0417 | $0.00232 |
| 4 | qwen-3-8-27b | 18/18 | 92.0% | 596/648 | 3.3s | 58.7s | $0.0377 | $0.00209 |
| 5 | z-ai-glm-5-3-flash | 18/18 | 89.7% | 581/648 | 16.7s | 300.8s | $0.0082 | $0.00045 |
| 6 | qwen-3-8-flash | 18/18 | 89.4% | 579/648 | 6.6s | 119.5s | $0.0062 | $0.00034 |


The full tool supports both `deepseek-v4-1-flash` through an API and `gemma4:12b` locally through Ollama and Tailscale. This notebook demonstrates the local Gemma model.

I chose Gemma 4 12B for its accuracy among the models I could run reliably on my hardware. DeepSeek V4.1 Flash was the most accurate API model in this benchmark and had a low cost per image.  


## Step 1: Extract the bet slip

A user sends a screenshot of a boosted parlay to the Telegram bot. The bot forwards the image to an image model. In this walkthrough, a locally hosted Gemma 4 12B model processes it through Ollama over Tailscale.

The Telegram integration is simulated here by passing the image directly to `ollama_process_image()`.

<img src="auto-devig-examples/5030624249995005052.jpg" alt="Example boosted parlay bet slip" width="450">

*Example bet slip used in the walkthrough.*

In [18]:
OLLAMA_HOST = os.getenv('OLLAMA_HOST')
MODEL = "gemma4:12b"
image_file = "5030624249995005052.jpg"
IMAGE = Path(f"auto-devig-examples/{image_file}")

results = ollama_process_image(ollama=OLLAMA_HOST, model=MODEL, image=IMAGE)

Leg 1: #5 Ohio State to Win (OSU @ IOWA)
   selection=Ohio State market='to Win' league='NCAAF' side=None line=None odds=None event='OSU @ IOWA'
Leg 2: Michigan to Win (MICH @ MINN)
   selection=Michigan market='to Win' league='NCAAF' side=None line=None odds=None event='MICH @ MINN'
Leg 3: #18 USC to Win (WAS @ USC)
   selection=USC market='to Win' league='NCAAF' side=None line=None odds=None event='WAS @ USC'


## Step 2: Search for a leg

The structured output contains the details needed to search for odds on each parlay leg. This walkthrough follows leg 2: "Michigan to Win"

In [19]:
leg = results["legs"][1]
print(f"{leg['selection']=}  {leg['market_raw']=}  {leg['event']=}  {leg['league'].lower()=}")

GAME = leg['event']
SEARCH_TEXT = leg['selection']
LEAGUE_ID = leg['league'].lower()
MARKET_RAW = leg['market_raw']

TOKEN = os.getenv('ODDS_SERVICE_TOKEN')
BOOKS = ["circa", "draftkings", "fanduel", "hard-rock", "novig", "pinnacle"]

df = search_for_event(api_token=TOKEN, books=BOOKS, search_text=SEARCH_TEXT, league_id=LEAGUE_ID)
print(f"\nInitial search found {len(df)} potential options\n")
df[["event","starts_at","league","event_id","book","market","period","selection", "side","line"]]

leg['selection']='Michigan'  leg['market_raw']='to Win'  leg['event']='MICH @ MINN'  leg['league'].lower()='ncaaf'

Initial search found 1351 potential options



,event,starts_at,league,event_id,book,market,period,selection,side,line
0,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,circa,Moneyline,None,Michigan Wolverines,NaN,NaN
1,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,circa,Point Spread,None,Michigan Wolverines,NaN,-5.5
2,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,draftkings,1st Half Last Team To Score,None,Michigan Wolverines,NaN,NaN
3,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,draftkings,1st Half Moneyline,None,Michigan Wolverines,NaN,NaN
4,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,draftkings,1st Half Player Touchdowns,None,Michigan Wolverines,Over,0.5
...,...,...,...,...,...,...,...,...,...,...
1346,Akron Zips @ Central Michigan Chippewas,2026-10-03 19:30:00+00:00,ncaaf,f91543eb-49c0-5a2a-9812-0377711ad09d,pinnacle,Point Spread,None,Central Michigan Chippewas,NaN,-6.5
1347,Akron Zips @ Central Michigan Chippewas,2026-10-03 19:30:00+00:00,ncaaf,f91543eb-49c0-5a2a-9812-0377711ad09d,pinnacle,Point Spread,None,Central Michigan Chippewas,NaN,-6.0
1348,Akron Zips @ Central Michigan Chippewas,2026-10-03 19:30:00+00:00,ncaaf,f91543eb-49c0-5a2a-9812-0377711ad09d,pinnacle,Point Spread,None,Central Michigan Chippewas,NaN,-5.5
1349,Akron Zips @ Central Michigan Chippewas,2026-10-03 19:30:00+00:00,ncaaf,f91543eb-49c0-5a2a-9812-0377711ad09d,pinnacle,Point Spread,None,Central Michigan Chippewas,NaN,-5.0


## Step 3: Resolve the market with Laya

The initial odds database search uses the selection extracted by Gemma 4. It deliberately returns a broad set of candidate events and markets. Laya, a locally hosted System 1 decision model, narrows these results with typed questions and returns choices with probability estimates.

For this football leg, the raw market label "to Win" refers to a full-game moneyline. The search can return dozens of different markets, Laya first predicts whether the label refers to a full game or a portion of one, such as a quarter or half. The code uses that prediction to filter the candidates, which is a critical step as the Laya model performance is dramatically degraded when many options are present. A second Laya question then selects the best market using descriptions from the odds database.

- `fetch_markets_glossary()` retrieves the available markets and their descriptions.
- `fetch_markets_and_descriptions()` keeps only glossary entries found in the Step 2 search results.

In [20]:
markets_glossary = fetch_markets_glossary(api_token=TOKEN, league_id=LEAGUE_ID)
print(f"Number of Total Markets: {len(markets_glossary)}")

markets = df.market.unique().tolist()
print(f"Number of Markets in df: {len(markets)}")

markets_and_descriptions = fetch_markets_and_descriptions(markets=markets, markets_glossary=markets_glossary)
print(f"Number of Matched Markets and Descriptions: {len(markets_and_descriptions)}")

Number of Total Markets: 50
Number of Markets in df: 22
Number of Matched Markets and Descriptions: 22


In [29]:
router = Router()
full_game = laya_noul_partial_game_bet(router=router, market_raw=MARKET_RAW)
print(f"\nIs the bet market for only an inning, quarter, or half of the game?  ({full_game['answers']['full_game']['noul']} out of 1.00)\n")

if full_game['answers']['full_game']['noul'] < 0.5:
    print(f"{MARKET_RAW=} is a full game bet.\n")
    filtered_markets_and_descriptions = {}
    for k,v in markets_and_descriptions.items():
        if "quarter" in k.lower().split(' '):
            continue
        elif "half" in k.lower().split(' '):
            continue
        elif "inning" in k.lower().split(' '):
            continue
        else:
            filtered_markets_and_descriptions[k] = v
else:
    print(f"{MARKET_RAW=} is bet for only a portion of the game.\n")
    filtered_markets_and_descriptions = {}
    for k,v in markets_and_descriptions.items():
        if "quarter" in k.lower().split(' '):
            filtered_markets_and_descriptions[k] = v
        elif "half" in k.lower().split(' '):
            filtered_markets_and_descriptions[k] = v
        elif "inning" in k.lower().split(' '):
            filtered_markets_and_descriptions[k] = v

print(f"Number of Matched Markets and Descriptions: {len(filtered_markets_and_descriptions)}")
print(f"Filtered Market Options: {list(filtered_markets_and_descriptions.keys())}\n")

laya_market = laya_process_search_results(router=router, market_raw=MARKET_RAW, markets=filtered_markets_and_descriptions)
MARKET = laya_market['answers']['market']['choice']
print(f"\nWhich market is the best match?  {MARKET=}  ({laya_market['answers']['market']['answer_confidence']} out of 1.00)")


df_by_market = df[df['market'] == MARKET]
print(f"\nNumber of Remaining Potential Options: {len(df_by_market)} \n")
df_by_market[["event","starts_at","league","event_id","book","market","period","selection", "side","line"]]


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1666.66it/s]



Is the bet market for only an inning, quarter, or half of the game?  (0.0691 out of 1.00)

MARKET_RAW='to Win' is a full game bet.

Number of Matched Markets and Descriptions: 6
Filtered Market Options: ['Moneyline', 'Point Spread', 'Player Touchdowns', 'Team Total Points', 'Winning Margin', 'Moneyline 3-Way']



Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1502.26it/s]



Which market is the best match?  MARKET='Moneyline'  (0.7449 out of 1.00)

Number of Remaining Potential Options: 29 



,event,starts_at,league,event_id,book,market,period,selection,side,line
0,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,circa,Moneyline,None,Michigan Wolverines,NaN,NaN
50,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,draftkings,Moneyline,None,Michigan Wolverines,NaN,NaN
153,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,fanduel,Moneyline,None,Michigan Wolverines,NaN,NaN
207,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,hard-rock,Moneyline,None,Michigan Wolverines,NaN,NaN
225,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,novig,Moneyline,None,Michigan Wolverines,NaN,NaN
259,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,pinnacle,Moneyline,None,Michigan Wolverines,NaN,NaN
269,Buffalo Bulls @ Western Michigan Bro...,2026-10-03 17:00:00+00:00,ncaaf,2baeab06-4e90-5c00-bf6a-d01f9866cd40,circa,Moneyline,None,Western Michigan Broncos,NaN,NaN
319,Buffalo Bulls @ Western Michigan Bro...,2026-10-03 17:00:00+00:00,ncaaf,2baeab06-4e90-5c00-bf6a-d01f9866cd40,draftkings,Moneyline,None,Western Michigan Broncos,NaN,NaN
422,Buffalo Bulls @ Western Michigan Bro...,2026-10-03 17:00:00+00:00,ncaaf,2baeab06-4e90-5c00-bf6a-d01f9866cd40,fanduel,Moneyline,None,Western Michigan Broncos,NaN,NaN
484,Buffalo Bulls @ Western Michigan Bro...,2026-10-03 17:00:00+00:00,ncaaf,2baeab06-4e90-5c00-bf6a-d01f9866cd40,hard-rock,Moneyline,None,Western Michigan Broncos,NaN,NaN


## Step 4: Resolve the event

Filtering by market substantially reduces the candidate rows, but several games and selections still contain "Michigan". Laya compares the abbreviated matchup from the screenshot with the candidate event names and selects the best match.

In [25]:
if len(df_by_market.event_id.unique()) == 1:
    EVENT_ID = df_by_market.event.unique()[0]
else:
    print(f"ERROR: MORE THAN ONE EVENT IDENTIFIED. {df_by_market.event.unique()}\n")

if len(df_by_market.selection.unique()) == 1:
    SELECTION = df_by_market.selection.unique()[0]
else:
    print(f"ERROR: MORE THAN ONE BET SELECTION IDENTIFIED. {df_by_market.selection.unique()}")

ERROR: MORE THAN ONE EVENT IDENTIFIED. <ArrowStringArray>
[   'Michigan Wolverines @ Minnesota Golden Gophers',
          'Buffalo Bulls @ Western Michigan Broncos',
 'Eastern Michigan Eagles @ Massachusetts Minutemen',
       'Michigan State Spartans @ Wisconsin Badgers',
           'Akron Zips @ Central Michigan Chippewas']
Length: 5, dtype: str

ERROR: MORE THAN ONE BET SELECTION IDENTIFIED. <ArrowStringArray>
[       'Michigan Wolverines',   'Western Michigan Broncos',
    'Eastern Michigan Eagles',    'Michigan State Spartans',
 'Central Michigan Chippewas']
Length: 5, dtype: str


In [11]:
print(f"{GAME=}")
print(f"{df.event.unique().tolist()}")
events = df.event.unique().tolist()
laya_event = laya_process_search_event(router=router, game=GAME, events=events)
print(f"laya_event: {laya_event['answers']['game']['choice']}  ({laya_event['answers']['game']['answer_confidence']} out of 1.00)")


EVENT = laya_event['answers']['game']['choice']
print(f"{EVENT=}")
df_final = df_by_market[df_by_market['event'] == EVENT]

if len(df_final.event_id.unique()) == 1:
    EVENT_ID = df_final.event_id.unique()[0]
else:
    print(f"ERROR: MORE THAN ONE EVENT ID IDENTIFIED. {df_final.event_id.unique()}")
if len(df_final.selection.unique()) == 1:
    SELECTION = df_final.selection.unique()[0]
else:
    print(f"ERROR: MORE THAN ONE MARKET SELECTION IDENTIFIED. {df_final.selection.unique()}")


df_final[["event","starts_at","league","event_id","book","market","period","selection", "side","line"]]

GAME='MICH @ MINN'
['Michigan Wolverines @ Minnesota Golden Gophers', 'Buffalo Bulls @ Western Michigan Broncos', 'Eastern Michigan Eagles @ Massachusetts Minutemen', 'Michigan State Spartans @ Wisconsin Badgers', 'Akron Zips @ Central Michigan Chippewas']
laya_event: Michigan Wolverines @ Minnesota Golden Gophers  (0.8295 out of 1.00)
EVENT='Michigan Wolverines @ Minnesota Golden Gophers'


,event,starts_at,league,event_id,book,market,period,selection,side,line
0,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,circa,Moneyline,None,Michigan Wolverines,NaN,NaN
50,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,draftkings,Moneyline,None,Michigan Wolverines,NaN,NaN
153,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,fanduel,Moneyline,None,Michigan Wolverines,NaN,NaN
207,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,hard-rock,Moneyline,None,Michigan Wolverines,NaN,NaN
226,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,novig,Moneyline,None,Michigan Wolverines,NaN,NaN
260,Michigan Wolverines @ Minnesota Gold...,2026-10-03 16:00:00+00:00,ncaaf,1a781503-6a54-571b-b719-84d079f53533,pinnacle,Moneyline,None,Michigan Wolverines,NaN,NaN


## Step 5: Retrieve both sides of the odds

Once the event and market are resolved, the remaining rows represent the same bet across sportsbooks. The code uses the event ID and market to request prices for both teams from the selected books. Both sides are needed to remove the sportsbook margin before calculating a leg's fair probability.

In [12]:
odds = fetch_leg_complete_odds(api_token=TOKEN, books=BOOKS, event_id=EVENT_ID, market=MARKET)
odds[["book", 'selection', 'price', ]]

,book,selection,price
0,circa,Michigan Wolverines,-215.0
1,circa,Minnesota Golden Gophers,185.0
2,draftkings,Michigan Wolverines,-225.0
3,draftkings,Minnesota Golden Gophers,185.0
4,fanduel,Michigan Wolverines,-215.0
5,fanduel,Minnesota Golden Gophers,176.0
6,hard-rock,Michigan Wolverines,-210.0
7,hard-rock,Minnesota Golden Gophers,175.0
8,novig,Michigan Wolverines,-203.0
9,novig,Minnesota Golden Gophers,194.0


### Final output

The full tool repeats this lookup for every parlay leg, calculates the parlay's expected value (EV), and returns the result through Telegram. This notebook demonstrates one leg and stops after retrieving its odds.

<img src="auto-devig-examples/result-example.png" alt="Example boosted parlay result" width="450">

*Example Telegram result from the full tool.*


Notes:
- This notebook is for demonstrational purposes and does not contain the exact production code
- The bot is publicly available @vvpunts_bot on telegram, the image devig feature may or may not be available to everyone.